# Entrenamiento en Kaggle (modelos clásicos, sin redes neuronales)

Antes de correr:
1. **Add Input** → busca `moltean/fruits` (Fruits-360) y agrégalo.
2. **Settings** → Internet: **On** (requiere teléfono verificado). No hace falta GPU: todo corre en CPU.
3. Para correr sin tener la pestaña abierta: **Save Version → Save & Run All**.

Este notebook solo clona el repo y llama a `src/`. Los cambios de código van en el repo, no aquí.

In [ ]:
!git clone -q https://github.com/11909/Clasificador-fruta-verdura.git repo
%cd repo
!pip install -q -r requirements.txt

Localiza las carpetas `Training` y `Test` del dataset adjunto y genera la config de esta corrida.

In [ ]:
import glob, yaml

train_dirs = sorted(glob.glob("/kaggle/input/**/Training", recursive=True))
test_dirs = sorted(glob.glob("/kaggle/input/**/Test", recursive=True))
print("train:", train_dirs)
print("test:", test_dirs)

cfg = yaml.safe_load(open("configs/baseline.yaml"))
cfg["data"]["train_dirs"] = train_dirs[:1]   # ajusta si adjuntas más de un dataset
cfg["data"]["test_dirs"] = test_dirs[:1]
cfg["data"]["max_por_clase"] = 60            # súbelo para usar más imágenes (más lento)
cfg["data"]["excluir_grupos"] = []           # ["nueces_semillas"] para sacar nueces y semillas
yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)

Revisa que todas las clases tengan etiqueta en `data/mapping.csv`.

In [ ]:
!python scripts/check_mapping.py {train_dirs[0]} | tail -5

Extrae características, compara los modelos con validación cruzada, reentrena el mejor y lo evalúa en test.

In [ ]:
!python -m src.train --config configs/run.yaml

Resultados: comparación de modelos (`resultados_cv.csv`), métricas en test y el modelo guardado (`modelo.joblib`) en `/kaggle/working/outputs/`.

In [ ]:
import csv, json
for fila in csv.DictReader(open("/kaggle/working/outputs/resultados_cv.csv")):
    print(f"{fila['modelo']:20} f1_macro={fila['f1_macro']}  ±{fila['f1_macro_std']}  ({fila['segundos']} s)")
print(json.dumps(json.load(open("/kaggle/working/outputs/test_metrics.json")), indent=2))